<a href="https://colab.research.google.com/github/vtu30294-ai/IHL-Usecases/blob/main/USECASE3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# SIMPLE FINE-TUNED CHATBOT
# ==========================================

!pip install -q transformers datasets sentencepiece accelerate

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

# ==========================================
# 1. CREATE DATASET
# ==========================================

data = {
    "question": [
        "What is Python?",
        "What is Python programming?",
        "Tell me about Python.",
        "Define Python.",

        "What is Artificial Intelligence?",
        "What is AI?",
        "Define Artificial Intelligence.",
        "Tell me about AI.",

        "What is Machine Learning?",
        "What is ML?",
        "Define Machine Learning.",
        "Tell me about Machine Learning.",

        "What is RAG?",
        "What does RAG mean?",
        "Define RAG.",
        "Tell me about RAG.",

        "What is Google Colab?",
        "What is Colab?",
        "Define Google Colab.",
        "Tell me about Google Colab.",

        "What is a chatbot?",
        "What does chatbot mean?",
        "Define chatbot.",
        "Tell me about chatbots."
    ],

    "answer": [
        "Python is a programming language.",
        "Python is a programming language.",
        "Python is a programming language.",
        "Python is a programming language.",

        "Artificial Intelligence enables computers to perform intelligent tasks.",
        "Artificial Intelligence enables computers to perform intelligent tasks.",
        "Artificial Intelligence enables computers to perform intelligent tasks.",
        "Artificial Intelligence enables computers to perform intelligent tasks.",

        "Machine Learning allows computers to learn from data.",
        "Machine Learning allows computers to learn from data.",
        "Machine Learning allows computers to learn from data.",
        "Machine Learning allows computers to learn from data.",

        "RAG means Retrieval Augmented Generation.",
        "RAG means Retrieval Augmented Generation.",
        "RAG means Retrieval Augmented Generation.",
        "RAG means Retrieval Augmented Generation.",

        "Google Colab is an online platform for running Python code.",
        "Google Colab is an online platform for running Python code.",
        "Google Colab is an online platform for running Python code.",
        "Google Colab is an online platform for running Python code.",

        "A chatbot is a computer program that communicates with users.",
        "A chatbot is a computer program that communicates with users.",
        "A chatbot is a computer program that communicates with users.",
        "A chatbot is a computer program that communicates with users."
    ]
}

dataset = Dataset.from_dict(data)

print("Dataset size:", len(dataset))

# ==========================================
# 2. LOAD MODEL
# ==========================================

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ==========================================
# 3. PREPARE DATA
# ==========================================

def prepare_data(examples):

    inputs = [
        "Question: " + q + " Answer:"
        for q in examples["question"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=64,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["answer"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_data = dataset.map(
    prepare_data,
    batched=True,
    remove_columns=dataset.column_names
)

# ==========================================
# 4. DATA COLLATOR
# ==========================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# ==========================================
# 5. TRAIN MODEL
# ==========================================

training_args = TrainingArguments(
    output_dir="./chatbot_model",

    num_train_epochs=10,

    per_device_train_batch_size=4,

    learning_rate=5e-5,

    logging_steps=1,

    save_strategy="no",

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator
)

print("Training started...")

trainer.train()

print("Training completed!")

# ==========================================
# 6. CHATBOT FUNCTION
# ==========================================

def chatbot(question):

    prompt = "Question: " + question + " Answer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        num_beams=4,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

# ==========================================
# 7. START CHATBOT
# ==========================================

print("\n================================")
print("       FINE-TUNED CHATBOT")
print("================================")
print("Ask a question.")
print("Type 'bye' to exit.\n")

while True:

    question = input("You: ")

    if question.lower() == "bye":
        print("Bot: Goodbye!")
        break

    answer = chatbot(question)

    print("Bot:", answer)

Dataset size: 24


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Training started...


Step,Training Loss
1,2.874000
2,2.650954
3,3.985490
4,2.575888
5,2.251543
6,2.306416
7,2.475908
8,2.284405
9,2.251644
10,2.352996


Training completed!

       FINE-TUNED CHATBOT
Ask a question.
Type 'bye' to exit.

Bot: Python is a programming language.
Bot: Artificial Intelligence (AI) is an intelligence technology.
Bot: RAG is a non-discriminational non-discriminational non-discriminational non-dis
